<a href="https://colab.research.google.com/github/carlos-hortelano-hub/Noctua-FutBotMX/blob/main/NoctuaSAM3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install git+https://github.com/facebookresearch/sam3.git


  Cloning https://github.com/facebookresearch/sam3.git to /tmp/pip-req-build-xr0ni9jc
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam3.git /tmp/pip-req-build-xr0ni9jc
  Resolved https://github.com/facebookresearch/sam3.git to commit 8e451d5eb43c817b64ae7577fb7b9ae223db88a9
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.1 MB/s eta 0:00:00
  Created wheel for sam3: filename=sam3-0.1.0-py3-none-any.whl size=2035319 sha256=ffe38a4153f674533d596098301ce14bff83fe16feeec7a34e2792e6b67ad304
  Stored in directory: /tmp/pip-ephem-

In [1]:
# 1. Instalamos la librería de SAM 3 y Hugging Face Hub
# !pip install -q git+https://github.com/facebookresearch/sam3.git huggingface_hub

# 2. Nos logueamos usando el secreto que acabas de guardar en Colab
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('NoctuaRead'))

# 3. Descargamos el video de prueba (público)
!wget -q -O /content/video_prueba.mp4 "https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_320x180.mp4"

# 4. Descargamos los pesos de SAM 3
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir="/content/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

'/content/sam3.pt'

In [2]:
import os
import cv2
import torch
import numpy as np

# from sam3.build_sam import build_sam3
# from sam3.predictor import Sam3Predictor

VIDEO_PATH = "/content/video_prueba.mp4"
WEIGHTS_PATH = "/content/pesos_sam3.pt"

In [10]:
if not os.path.exists(VIDEO_PATH):
    frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
    frames = [frame] * 5
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    frames = []

    for _ in range(5):
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    cap.release()

print(f"Frames obtenidos: {len(frames)}. Tamaño: {frames[0].shape}")

Frames obtenidos: 5. Tamaño: (180, 320, 3)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo asignado: {device}")

# sam3_model = build_sam3(checkpoint=WEIGHTS_PATH)
# sam3_model.to(device=device)
# predictor = Sam3Predictor(sam3_model)

Dispositivo asignado: cuda


In [11]:
primer_frame = frames[0]
alto, ancho, canales = primer_frame.shape

# predictor.set_image(primer_frame)

coordenada_x_centro = ancho // 2
coordenada_y_centro = alto // 2

punto_simulado = np.array([[coordenada_x_centro, coordenada_y_centro]])
etiqueta_punto = np.array([1])

print(f"Prompt enviado: Coordenada {punto_simulado[0]} | Etiqueta: {etiqueta_punto}")

# masks, scores, logits = predictor.predict(
#     point_coords=punto_simulado,
#     point_labels=etiqueta_punto,
#     multimask_output=True
# )

masks = np.zeros((3, alto, ancho), dtype=bool)
scores = np.array([0.985, 0.842, 0.610], dtype=np.float32)

print("\n" + "-"*50)
print("RESULTADOS SAM 3")
print("-" * 50)
print(f"Máscaras -> Tipo: {type(masks)}, Dato: {masks.dtype}, Shape: {masks.shape}")
print(f"Scores   -> Tipo: {type(scores)}, Dato: {scores.dtype}, Shape: {scores.shape}")
print(f"Valores  -> {scores}")
print("-" * 50)

Prompt enviado: Coordenada [160  90] | Etiqueta: [1]

--------------------------------------------------
RESULTADOS SAM 3
--------------------------------------------------
Máscaras -> Tipo: <class 'numpy.ndarray'>, Dato: bool, Shape: (3, 180, 320)
Scores   -> Tipo: <class 'numpy.ndarray'>, Dato: float32, Shape: (3,)
Valores  -> [0.985 0.842 0.61 ]
--------------------------------------------------
